# 3 — Controllo: l'integratore e la funzione di reiezione

Banco PRIMA: sensore a piramide modulato + DM Multi-DM 140 (96 attuatori nella pupilla).
In questo notebook studiamo la **legge di controllo** del loop: un integratore modale con
guadagno $g$ e *leak* $\ell$, e un ritardo di $D$ frame tra la misura e l'applicazione
della correzione.

**Obiettivi**
1. ricavare la funzione di trasferimento di reiezione (RTF) del loop e la sua stabilità;
2. misurarla dalla telemetria e confrontarla con il modello;
3. stimare il ritardo $D$ del sistema e scegliere il guadagno ottimo.

Lavoriamo prima sul **gemello numerico** del banco (stesso software, stessa interfaccia),
poi ripetiamo le misure sul banco vero (ultima sezione).

**Convenzioni.** Tutte le grandezze modali sono coefficienti dei modi KL del ricostruttore
(rms unitario sugli attuatori attivi), convertiti in nm di fronte d'onda *nominali*.
Il tempo è in frame: $f$ è la frequenza normalizzata, $f/f_s \in [0, 1/2]$.

Sul banco la turbolenza non arriva dall'esterno: la **genera il DM stesso**, sommando al
comando del loop uno schermo di fase che scorre (*frozen flow*). Quindi il disturbo è
**noto**, frame per frame: un lusso che al telescopio non si ha.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

from prima.aolab import lab
from prima.aolab.analysis import PrimaAnalyzer, list_tags

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True

## 1. Il modello

A ogni frame $k$ il ricostruttore $R$ trasforma le slope $s_k$ nei coefficienti modali
misurati $m_k = R\,s_k$, e l'integratore aggiorna il comando (in coefficienti modali):

$$ c_k = \ell\, c_{k-1} - g\, m_k . $$

Il DM somma il comando del loop $c$ al disturbo $d$ (generato anch'esso dal DM). Il sensore
vede la somma con un ritardo di $D$ frame (lettura della camera, calcolo, tempo in cui il
comando arriva al DM mentre la camera sta già esponendo):

$$ m_k = d_{k-D} + c_{k-D} + n_k , $$

dove $n_k$ è il rumore di misura.

**Esercizio 1.**
(a) Passando alla trasformata $z$, ricavate la funzione di reiezione
$H_\mathrm{rej}(z) = M(z) / \big(z^{-D} D(z)\big)$ (residuo misurato / disturbo) e la
funzione di trasferimento del rumore $H_n(z) = C(z)/N(z)$.
(b) Scrivete una funzione `rtf(f, g, leak=1.0, D=1)` che restituisca $|H_\mathrm{rej}|^2$
per frequenze normalizzate $f$ e tracciatela, in scala log-log, per $g = 0.2, 0.5, 1, 1.5$
con $D=1$, e poi con $D=2$.
(c) Che cosa succede a bassa frequenza al variare del leak? E vicino a Nyquist?

In [ ]:
def rtf(f, g, leak=1.0, D=1):
    """|H_rej|^2 alle frequenze normalizzate f (= f/fs)."""
    z = np.exp(2j * np.pi * f)
    # TODO
    raise NotImplementedError

f = np.linspace(1e-3, 0.5, 2000)
# TODO: grafici per D=1 e D=2

## 2. Stabilità

**Esercizio 2.** I poli del sistema ad anello chiuso sono le radici di
$z^{D-1}(z-\ell) + g = 0$. Calcolate numericamente (con `np.roots`) il guadagno massimo
stabile $g_\mathrm{max}$ per $D = 1, 2, 3, 4$ e $\ell = 1$. Confrontate con il risultato
analitico per $D=1$ e $D=2$. Qual è la **banda di reiezione** (frequenza dove $|H|^2 = 1$)
a $g = g_\mathrm{max}/2$?

In [ ]:
# TODO

## 3. La misura: telemetria dal simulatore

Avviamo il gemello numerico del banco già calibrato (pupille, riferimento, matrice di
interazione con 50 modi KL, ricostruttore). Si usa esattamente come il banco vero:
`e` è lo stesso `AOEngine` che c'è dietro la GUI.

| comando | effetto |
|---|---|
| `e.set_turbulence(True, rms=0.01, wind=0.5, seed=1)` | turbolenza sul DM: `rms` in unità DM (×18000 ≈ nm), `wind` in campioni/frame |
| `e.lp.gain`, `e.lp.leak` | parametri dell'integratore |
| `e.close_loop()`, `e.open_loop(reset=True)` | chiude / apre (e azzera) il loop |
| `lab.record(e, N, 'etichetta')` | registra N frame e restituisce un `PrimaAnalyzer` |

Ogni registrazione finisce in `~/git/prima_data/telemetry/<tag>/` e si può riaprire con
`PrimaAnalyzer.get(tag)`: vedi `list_tags()`.

In [ ]:
e = lab.sim_engine()
e.set_turbulence(True, rms=0.01, wind=0.5, seed=1)
lab.settle(e, 50)
print('loop rate %.0f Hz (simulatore), %d modi' % (e.loop_rate_hz(), e.rec.n_modes))

Prima registrazione: **anello aperto**. Il residuo misurato è il disturbo (ritardato) più
il rumore.

In [ ]:
e.open_loop(reset=True)
lab.settle(e, 20)
ol = lab.record(e, 2048, 'ctrl-ol')
ol.summary()

Il `PrimaAnalyzer` espone la telemetria come serie temporali di `arte`:

- `an.loop.residual_modes` — $m_k$ (tempo × modo, nm)
- `an.loop.integrated_modes` — $c_k$
- `an.loop.disturbance_modes` — $d_k$, il disturbo vero proiettato sui modi
- `an.loop.pseudo_open_loop_modes` — $m_k - c_{k-D}$ (con `an.delay_frames`)
- `an.wfs.slopes`, `an.dm.commands`, `an.calib.im`, `an.meta`, …

`.get_data()` restituisce l'array (una `Quantity` di astropy: `.value` per il numpy puro),
`.time_std`, `.power()`, `.plot_spectra()` fanno statistiche e spettri.

In [ ]:
m = ol.loop.residual_modes.get_data().value
d = ol.loop.disturbance_modes.get_data().value
plt.plot(ol.time, m[:, 0], label='misurato m (modo 0, tip)')
plt.plot(ol.time, d[:, 0], label='disturbo d (modo 0)')
plt.xlabel('t [s]'); plt.ylabel('nm'); plt.legend(); plt.show()

**Esercizio 3 — il ritardo.** In anello aperto $m_k = d_{k-D} + n_k$.
Usate la telemetria (`m`, `d`) per stimare $D$: per esempio con la cross-correlazione,
oppure trovando lo shift che minimizza $\mathrm{rms}(m_k - d_{k-D})$. Attenzione al termine
costante: il WFS vede anche l'aberrazione statica del banco.
Quanto vale il rumore $n_k$ (rms totale in nm)?

In [ ]:
# TODO

Ora chiudiamo il loop con diversi guadagni (leak = 1, integratore puro) e registriamo.
Per ogni registrazione la **RTF misurata** è il rapporto tra lo spettro del residuo e lo
spettro del disturbo in anello aperto, che stimiamo con il *pseudo open loop*
$m_k - c_{k-D}$ (lo stesso che si fa al telescopio, dove $d$ non è noto).

**Esercizio 4.**
(a) Registrate 2048 frame per $g = 0.2, 0.5, 1.0, 1.5$ (lasciate assestare il loop prima
di registrare: `lab.settle(e, 100)`).
(b) Calcolate le PSD (Welch, per esempio `nperseg=256`, media sui modi) di residuo e pseudo
open loop, il loro rapporto, e confrontatelo con `rtf(f, g, D=...)` dell'esercizio 1.
(c) Verificate che lo pseudo open loop coincida con il disturbo vero $d$.

In [ ]:
gains = [0.2, 0.5, 1.0, 1.5]
runs = {}
for g in gains:
    pass   # TODO: imposta gain/leak, chiudi, assesta, registra in runs[g]

## 4. Il guadagno ottimo

Il residuo ha due contributi con andamento opposto in $g$: l'errore **temporale**
(disturbo non inseguito, diminuisce con $g$) e il **rumore** propagato (cresce con $g$).
Finora il simulatore aveva moltissima luce (6·10⁶ fotoni per frame) e quasi niente rumore.
Portiamolo alla luce del banco a 41 Hz: `e.sim_model.flux = 2e4`.

**Esercizio 5.**
(a) Rumore puro: spegnete la turbolenza, registrate in anello aperto e stimate $\sigma_n$
(rms totale per frame, nm). Poi chiudete il loop con $g = 0.2, 0.5, 1, 1.5$ e mostrate che,
per $D=1$ e $\ell=1$, la varianza del residuo *misurato* vale $\sigma_n^2 \cdot 2/(2-g)$
e quella del residuo *vero* sul DM ($d+c$) vale $\sigma_n^2 \cdot g/(2-g)$.
(b) Con la turbolenza, misurate il residuo rms misurato e vero in funzione di $g$ tra 0.05
e 1.8, per `wind=0.1` e `wind=2.0` campioni/frame (1024 frame per punto bastano).
(c) Dove sta il guadagno ottimo nei due casi? Perché si sposta? Quale dei due residui
(misurato o vero) usereste per sceglierlo, e perché?

In [ ]:
e.sim_model.flux = 2e4
# TODO

**Domande.**
- Il residuo *misurato* dal WFS non è il residuo *vero* del fronte d'onda: in che cosa
  differiscono? (Suggerimento: `an.dm.residual_commands` è disturbo + correzione sul DM,
  indipendente dal rumore del sensore.)
- Per il solo rumore il residuo misurato è $\sqrt{2/g}$ volte quello vero. Per l'errore
  temporale invece misurato e vero coincidono (a meno del ritardo): perché?
- Con $D=2$ come cambierebbero i risultati?

In [ ]:
lab.close(e)

## 5. Sul banco

Ripetete le misure principali sul banco vero, a 41 Hz (modulazione 50 Hz, esposizione 20 ms).

1. Avviate la GUI dall'icona **PRIMA AO lab GUI** e seguite la sequenza
   *Find pupils → ROI on pupils → Reference = now → Acquire IM → Compute reconstructor*.
2. Accendete la turbulenza dal riquadro *Disturbance* (rms 150 nm, wind 0.3).
3. Nel riquadro *Loop control* registrate (pulsante **RECORD**, 2048 frame, ≈ 50 s)
   una sequenza in anello aperto e almeno tre guadagni, con etichette chiare
   (es. `ol`, `g0.2`, `g0.5`, `g1.0`).
4. Analizzate le registrazioni qui sotto con `PrimaAnalyzer.get(tag)`.

**Esercizio 6.** Quanto vale $D$ sul banco? Qual è $g_\mathrm{max}$? Confrontate la RTF
misurata con il modello: dove si discosta, e perché? (Pensate a che cosa fa la camera
mentre il DM si muove, e all'esposizione di 20 ms su un periodo di 24 ms.)

In [ ]:
list_tags('*-ol')[-5:]        # le ultime registrazioni in anello aperto

In [ ]:
# bench = {g: PrimaAnalyzer.get('...') for g in (...)}
# TODO